# Đề 9 — E-Commerce Customer Segmentation

## Notebook 01 — Khảo sát và Làm sạch dữ liệu

**Bộ dữ liệu:** Online Retail II (UCI Machine Learning Repository, ID 502)
Giao dịch của một nhà bán lẻ trực tuyến tại Anh, từ 01/12/2009 đến 09/12/2011.

**Mục tiêu notebook này**

1. Nạp dữ liệu từ hai sheet của file Excel và hợp nhất
2. Khảo sát chất lượng dữ liệu thô, xác định các vấn đề cần xử lý
3. Làm sạch theo pipeline có ghi nhật ký, phục vụ tính tái lập
4. Xuất `retail_clean.parquet` làm đầu vào cho các notebook tiếp theo

**Đầu ra:** `data/retail_raw.parquet` và `data/retail_clean.parquet`

---

## 0. Thiết lập

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DATA      = Path('data')
XLSX      = DATA / 'online_retail_II.xlsx'
RAW_CACHE = DATA / 'retail_raw.parquet'
CLEAN_OUT = DATA / 'retail_clean.parquet'

print('pandas', pd.__version__, '| numpy', np.__version__)

pandas 3.0.5 | numpy 2.5.3


---

## 1. Nạp dữ liệu

### 1.1 Kiểm tra file nguồn

In [2]:
print('File tồn tại :', XLSX.exists())
if XLSX.exists():
    print(f'Kích thước   : {XLSX.stat().st_size / 1024**2:.1f} MB')
    print('Các sheet    :', pd.ExcelFile(XLSX, engine='openpyxl').sheet_names)

File tồn tại : True
Kích thước   : 43.5 MB


Các sheet    : ['Year 2009-2010', 'Year 2010-2011']


### 1.2 Đọc, hợp nhất và ép kiểu

File gốc có **hai sheet** (`Year 2009-2010` và `Year 2010-2011`). Phải đọc cả hai
rồi nối lại, nếu không sẽ mất một nửa dữ liệu mà không có cảnh báo nào.

**Vấn đề kiểu dữ liệu.** Ba cột bị kiểu hỗn hợp trong file gốc:

| Cột | Vấn đề | Xử lý |
|---|---|---|
| `Invoice` | lẫn số (`489434`) và chuỗi (`C489449` — đơn huỷ) | ép về `str` |
| `StockCode` | lẫn số (`22423`) và chuỗi (`85123A`, `POST`) | ép về `str` |
| `Description` | lẫn số giữa các chuỗi mô tả, **và có giá trị thiếu** | ép về `string` (nullable) |

`Description` dùng `string` thay vì `str` vì nó có `NaN`: dùng `astype(str)` sẽ biến
`NaN` thành chuỗi `'nan'`, làm mất dấu vết giá trị thiếu.

**Cache.** Đọc file `.xlsx` mất khoảng 65 giây vì `openpyxl` phải phân tích từng ô
như một phần tử XML. Kết quả được lưu ra `.parquet` (định dạng nhị phân, nén theo cột)
nên các lần chạy sau chỉ mất dưới 1 giây.

In [3]:
def load_raw(force_reload: bool = False) -> pd.DataFrame:
    """Nạp dữ liệu thô. Ưu tiên đọc cache parquet, nhanh hơn khoảng 60 lần."""
    if RAW_CACHE.exists() and not force_reload:
        print(f'Đọc từ cache: {RAW_CACHE.name}')
        return pd.read_parquet(RAW_CACHE)

    print(f'Đọc từ Excel: {XLSX.name} (mất 1-3 phút)')
    t0 = time.time()
    sheets = pd.read_excel(XLSX, sheet_name=None, engine='openpyxl')
    print(f'  Hoàn tất sau {time.time() - t0:.0f} giây')
    for name, s in sheets.items():
        print(f'  {name}: {len(s):,} dòng')

    data = pd.concat(
        [s.assign(SourceSheet=name) for name, s in sheets.items()],
        ignore_index=True,
    )

    # Ba cột kiểu hỗn hợp -- xem giải thích ở markdown phía trên
    data['Invoice']     = data['Invoice'].astype(str)
    data['StockCode']   = data['StockCode'].astype(str)
    data['Description'] = data['Description'].astype('string')

    data.to_parquet(RAW_CACHE, index=False)
    print(f'  Đã lưu cache: {RAW_CACHE.name}')
    return data


df = load_raw()
print(f'\nTổng: {len(df):,} dòng x {df.shape[1]} cột')

Đọc từ cache: retail_raw.parquet

Tổng: 1,067,371 dòng x 9 cột


---

## 2. Khảo sát dữ liệu thô

### 2.1 Cấu trúc

In [4]:
print(df.dtypes.to_string())
print()
df.head()

Invoice                   str
StockCode                 str
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
SourceSheet               str



,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


### 2.2 Giá trị thiếu

In [5]:
missing = df.isna().sum()
missing = missing[missing > 0]

print('Số giá trị thiếu theo cột:')
print(missing.to_string())
print()
print(f"Customer ID thiếu: {df['Customer ID'].isna().mean():.1%} tổng số dòng")

Số giá trị thiếu theo cột:
Description      4382
Customer ID    243007

Customer ID thiếu: 22.8% tổng số dòng


### 2.3 Khoảng thời gian và phần chồng lấn giữa hai sheet

Ngày cuối cùng quyết định **ngày tham chiếu** khi tính Recency ở notebook sau.

Cần kiểm tra thêm khoảng thời gian của **từng sheet**: nếu hai sheet phủ lên nhau,
các giao dịch trong phần chung sẽ bị đếm hai lần sau khi nối.

In [6]:
print('Toàn bộ dữ liệu:')
print('  Từ :', df['InvoiceDate'].min())
print('  Đến:', df['InvoiceDate'].max())
print()
print(df.groupby('SourceSheet')['InvoiceDate'].agg(['min', 'max', 'size']).to_string())

Toàn bộ dữ liệu:
  Từ : 2009-12-01 07:45:00
  Đến: 2011-12-09 12:50:00

                               min                 max    size
SourceSheet                                                   
Year 2009-2010 2009-12-01 07:45:00 2010-12-09 20:01:00  525461
Year 2010-2011 2010-12-01 08:26:00 2011-12-09 12:50:00  541910


In [7]:
SHEET_OLD, SHEET_NEW = 'Year 2009-2010', 'Year 2010-2011'

old_inv = set(df.loc[df['SourceSheet'] == SHEET_OLD, 'Invoice'])
new_inv = set(df.loc[df['SourceSheet'] == SHEET_NEW, 'Invoice'])
shared  = old_inv & new_inv

in_shared = df['Invoice'].isin(shared)
cols = [c for c in df.columns if c != 'SourceSheet']
part_old = df[in_shared & (df['SourceSheet'] == SHEET_OLD)][cols].sort_values(cols).reset_index(drop=True)
part_new = df[in_shared & (df['SourceSheet'] == SHEET_NEW)][cols].sort_values(cols).reset_index(drop=True)

print(f'Hóa đơn xuất hiện ở cả hai sheet: {len(shared):,}')
print(f'Số dòng của các hóa đơn đó      : {len(part_old):,} ở mỗi sheet')
print(f'Hai phần giống hệt nhau         : {part_old.equals(part_new)}')
print(f'Khoảng thời gian chồng lấn      : {part_old["InvoiceDate"].min().date()} → {part_old["InvoiceDate"].max().date()}')

# Đánh dấu bản lặp ở sheet sau, dùng lại ở mục 2.5 và mục 3
dup_sheet = in_shared & (df['SourceSheet'] == SHEET_NEW)

Hóa đơn xuất hiện ở cả hai sheet: 1,088
Số dòng của các hóa đơn đó      : 22,523 ở mỗi sheet
Hai phần giống hệt nhau         : True
Khoảng thời gian chồng lấn      : 2010-12-01 → 2010-12-09


**Kết luận:** hai sheet **chồng lấn 9 ngày** (01–09/12/2010) và phần chồng lấn
giống hệt nhau từng dòng. Nếu không xử lý, mọi giao dịch trong 9 ngày này bị tính
**hai lần** vào Monetary.

Bước `drop_duplicates()` thông thường **không bắt được** lỗi này: cột `SourceSheet`
khác nhau nên pandas coi hai bản là hai dòng khác nhau. Mục 3 xử lý bằng một bước
riêng: bỏ các hóa đơn của sheet sau đã có ở sheet trước.

Ngày tham chiếu cho Recency = ngày cuối + 1 ngày.

### 2.4 Đơn huỷ và giá trị bất thường

In [8]:
is_cancel = df['Invoice'].str.startswith('C')
neg_qty   = df['Quantity'] <= 0
bad_price = df['Price'] <= 0

print(f'Đơn huỷ (Invoice bắt đầu C) : {is_cancel.sum():>7,} dòng ({is_cancel.mean():.2%})')
print(f'Quantity <= 0               : {neg_qty.sum():>7,} dòng')
print(f'Price <= 0                  : {bad_price.sum():>7,} dòng')
print()
print(f'Quantity <= 0 NHƯNG không phải đơn huỷ: {(neg_qty & ~is_cancel).sum():,} dòng')
print('   => lọc đơn huỷ thôi chưa đủ, phải lọc cả hai điều kiện')

Đơn huỷ (Invoice bắt đầu C) :  19,494 dòng (1.83%)
Quantity <= 0               :  22,950 dòng
Price <= 0                  :   6,207 dòng

Quantity <= 0 NHƯNG không phải đơn huỷ: 3,457 dòng
   => lọc đơn huỷ thôi chưa đủ, phải lọc cả hai điều kiện


### 2.5 Đơn đặt rồi huỷ: bỏ dòng huỷ thôi là chưa đủ

Một dòng huỷ (`Invoice` bắt đầu bằng `C`) là bút toán **đảo** của một đơn mua trước đó.
Nếu chỉ bỏ dòng huỷ mà giữ lại đơn gốc, khách hàng vẫn được tính là đã chi tiền cho
món hàng mà họ thực ra đã huỷ.

Hai khách hàng dưới đây cho thấy mức độ nghiêm trọng:

In [9]:
show = ['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'InvoiceDate']

for cid in [16446, 12346]:
    big = df[(df['Customer ID'] == cid) & (df['Quantity'].abs() > 10_000)]
    print(f'Khách {cid}:')
    print(big[show].to_string(index=False))
    print()

Khách 16446:
Invoice StockCode                 Description  Quantity  Price         InvoiceDate
 581483     23843 PAPER CRAFT , LITTLE BIRDIE     80995   2.08 2011-12-09 09:15:00
C581484     23843 PAPER CRAFT , LITTLE BIRDIE    -80995   2.08 2011-12-09 09:27:00

Khách 12346:
Invoice StockCode                    Description  Quantity  Price         InvoiceDate
 541431     23166 MEDIUM CERAMIC TOP STORAGE JAR     74215   1.04 2011-01-18 10:01:00
C541433     23166 MEDIUM CERAMIC TOP STORAGE JAR    -74215   1.04 2011-01-18 10:17:00



Khách **16446** đặt 80.995 sản phẩm (≈ 168.000 bảng) rồi huỷ sau 12 phút. Khách **12346**
đặt 74.215 sản phẩm (≈ 77.000 bảng) rồi huỷ sau 16 phút. Nếu chỉ bỏ dòng huỷ, cả hai
trở thành khách "siêu VIP" trong khi thực tế gần như không chi tiêu.

Đây là **outlier giả do dữ liệu**, khác với outlier thật (khách bán buôn mua nhiều thật).
Outlier giả phải được sửa ngay ở bước làm sạch. Outlier thật thì giữ lại và xử lý ở bước
phân cụm (notebook 03, mục 4.2).

**Cách khớp:** mỗi dòng huỷ được ghép với **một** đơn mua có cùng khách hàng, cùng mã sản
phẩm, cùng số lượng, và diễn ra **trước** thời điểm huỷ. Nếu có nhiều đơn khớp, chọn đơn
gần nhất. Mỗi đơn mua chỉ được ghép một lần.

In [10]:
def match_cancellations(data: pd.DataFrame) -> pd.DataFrame:
    """Ghép mỗi dòng huỷ với đơn mua gốc của nó.

    Khớp khi: cùng Customer ID, cùng StockCode, cùng số lượng, đơn mua diễn ra
    trước (hoặc cùng lúc) dòng huỷ. Chọn đơn mua gần nhất; mỗi đơn mua chỉ ghép một lần.
    Trả về bảng hai cột: chỉ số dòng huỷ và chỉ số dòng đơn mua gốc.
    """
    key  = ['Customer ID', 'StockCode', 'Qty']
    cols = ['InvoiceDate', 'Customer ID', 'StockCode', 'Quantity']
    is_c = data['Invoice'].str.startswith('C')

    buys = (data.loc[~is_c & (data['Quantity'] > 0), cols]
            .rename(columns={'Quantity': 'Qty'})
            .rename_axis('buy_idx').reset_index())
    pending = (data.loc[is_c & (data['Quantity'] < 0), cols]
               .assign(Qty=lambda x: -x['Quantity']).drop(columns='Quantity')
               .rename_axis('cancel_idx').reset_index())

    pairs = []
    while len(pending):
        # Với mỗi dòng huỷ, tìm đơn mua cùng khóa, gần nhất về trước
        m = pd.merge_asof(pending.sort_values('InvoiceDate'), buys.sort_values('InvoiceDate'),
                          on='InvoiceDate', by=key, direction='backward')
        # Nhiều dòng huỷ có thể cùng trỏ vào một đơn mua -> chỉ giữ một, phần còn lại thử lại vòng sau
        m = m.dropna(subset=['buy_idx']).drop_duplicates('buy_idx')
        if m.empty:
            break
        pairs.append(m[['cancel_idx', 'buy_idx']].astype('int64'))
        buys    = buys[~buys['buy_idx'].isin(m['buy_idx'])]
        pending = pending[~pending['cancel_idx'].isin(m['cancel_idx'])]

    return (pd.concat(pairs, ignore_index=True) if pairs
            else pd.DataFrame({'cancel_idx': [], 'buy_idx': []}, dtype='int64'))


has_id = df[~dup_sheet & df['Customer ID'].notna()]   # đã bỏ phần chồng lấn ở mục 2.3
pairs  = match_cancellations(has_id)

cancels  = has_id[has_id['Invoice'].str.startswith('C') & (has_id['Quantity'] < 0)]
c_value  = -(cancels['Quantity'] * cancels['Price'])
buys_hit = has_id.loc[pairs['buy_idx']]
fake_rev = (buys_hit['Quantity'] * buys_hit['Price']).sum()

print(f'Dòng huỷ (có Customer ID)   : {len(cancels):,}')
print(f'Ghép được với đơn gốc       : {len(pairs):,} dòng ({len(pairs) / len(cancels):.1%} số dòng huỷ, '
      f'{c_value.loc[pairs["cancel_idx"]].sum() / c_value.sum():.1%} giá trị huỷ)')
print(f'Doanh thu ảo nếu giữ đơn gốc: {fake_rev:,.0f} bảng')
print()
print('5 đơn gốc lớn nhất bị huỷ:')
print(buys_hit.assign(Value=buys_hit['Quantity'] * buys_hit['Price'])
      .nlargest(5, 'Value')[['Customer ID', 'StockCode', 'Description', 'Quantity', 'Price', 'Value']]
      .to_string(index=False))

Dòng huỷ (có Customer ID)   : 18,446
Ghép được với đơn gốc       : 6,430 dòng (34.9% số dòng huỷ, 60.2% giá trị huỷ)
Doanh thu ảo nếu giữ đơn gốc: 712,074 bảng

5 đơn gốc lớn nhất bị huỷ:
 Customer ID StockCode                         Description  Quantity    Price    Value
     16446.0     23843         PAPER CRAFT , LITTLE BIRDIE     80995     2.08 168469.6
     12346.0     23166      MEDIUM CERAMIC TOP STORAGE JAR     74215     1.04  77183.6
     15098.0     22502      PICNIC BASKET WICKER 60 PIECES        60   649.50  38970.0
     15838.0     84347 ROTATING SILVER ANGELS T-LIGHT HLDR      9360     1.69  15818.4
     12918.0         M                              Manual         1 10953.50  10953.5


Chỉ khoảng 35% **số dòng** huỷ ghép được, nhưng chúng chiếm phần lớn **giá trị** huỷ: các
đơn lớn bị huỷ gần như luôn bị huỷ nguyên đơn.

Các dòng huỷ còn lại **không ghép được** vì hai lý do: huỷ **một phần** đơn (số lượng
huỷ nhỏ hơn số lượng mua), hoặc đơn gốc nằm **trước tháng 12/2009**, ngoài phạm vi dữ
liệu. Các trường hợp này giữ nguyên và ghi nhận là một hạn chế. Ghép theo số lượng
chính xác đã bắt được đúng các ca "đặt nhầm rồi huỷ ngay", vốn là nguồn tạo outlier giả.

### 2.6 Số lượng giá trị duy nhất và thống kê mô tả

In [11]:
for col in ['Invoice', 'StockCode', 'Customer ID', 'Country']:
    print(f'{col:<13} {df[col].nunique():>8,}')

stats = df[['Quantity', 'Price']].describe()
display(stats.style.format('{:,.2f}'))

Invoice         53,628
StockCode        5,305
Customer ID      5,942
Country             43


,Quantity,Price
count,"1,067,371.00","1,067,371.00"
mean,9.94,4.65
std,172.71,123.55
min,"-80,995.00","-53,594.36"
25%,1.00,1.25
50%,3.00,2.10
75%,10.00,4.15
max,"80,995.00","38,970.00"


### 2.7 Phân tích mã sản phẩm

Mã sản phẩm chuẩn có dạng **5 chữ số, đôi khi kèm chữ cái** (`85123A`).
Các mã không khớp dạng này cần được kiểm tra trước khi quyết định loại bỏ.

In [12]:
std_pattern = df['StockCode'].str.match(r'^\d{5}[A-Za-z]*$')

print(f'Dòng có mã không chuẩn: {(~std_pattern).sum():,}')
print(f'Số mã không chuẩn     : {df.loc[~std_pattern, "StockCode"].nunique()}')
print()
print(df.loc[~std_pattern, 'StockCode'].value_counts().head(20).to_string())

Dòng có mã không chuẩn: 6,094
Số mã không chuẩn     : 63

StockCode
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
gift_0001_10      16
CRUK              16
DCGS0076          15
TEST001           15
DCGS0003          14


Danh sách trên cho thấy phần lớn là **phí và bút toán**, không phải hàng hóa:

| Nhóm | Mã |
|---|---|
| Phí vận chuyển | `POST`, `DOT`, `C2` |
| Điều chỉnh thủ công | `M`, `m`, `ADJUST`, `B` (bad debt) |
| Chiết khấu, hàng mẫu | `D`, `S` |
| Phí dịch vụ | `BANK CHARGES`, `AMAZONFEE`, `CRUK` |
| Phiếu quà tặng | `gift_0001_*` |
| Dòng kiểm thử | `TEST001` |

Riêng nhóm `DCGS*` cần kiểm tra thêm vì chưa rõ bản chất.

In [13]:
dcgs = df[df['StockCode'].str.startswith('DCGS')]

print(f'{len(dcgs)} dòng, {dcgs["StockCode"].nunique()} mã')
print()
print(dcgs[['StockCode', 'Description', 'Price']]
      .dropna()
      .drop_duplicates('StockCode')
      .head(10)
      .to_string(index=False))

158 dòng, 34 mã

StockCode                  Description  Price
 DCGS0058             MISO PRETTY  GUM   0.83
 DCGS0068            DOGS NIGHT COLLAR   8.65
 DCGS0004   HAYNES CAMPER SHOULDER BAG  17.35
 DCGS0076 SUNJAR LED NIGHT NIGHT LIGHT  16.48
 DCGS0003          BOXED GLASS ASHTRAY   2.57
 DCGS0072       CAT CAMOUFLAGUE COLLAR   5.17
 DCGS0044      HANDZ-OFF CAR FRESHENER   2.57
DCGS0066N      NAVY CUDDLES DOG HOODIE   8.65
 DCGS0069        OOH LA LA DOGS COLLAR  16.13
 DCGS0070        CAMOUFLAGE DOG COLLAR  12.72


**Kết luận:** `DCGS*` là **sản phẩm thật** (`MISO PRETTY GUM`, `DOGS NIGHT COLLAR`,
`SUNJAR LED NIGHT LIGHT`...), chỉ dùng hệ mã khác.

Nếu áp dụng regex chặt `^\d{5}[A-Za-z]*$` một cách máy móc, ta sẽ loại bỏ nhầm
158 dòng sản phẩm hợp lệ. Bộ lọc ở mục 3 được điều chỉnh để giữ lại nhóm này.

### 2.8 Tổng hợp các vấn đề phát hiện được

| # | Vấn đề | Quy mô | Ảnh hưởng nếu không xử lý |
|---|---|---|---|
| 1 | `Customer ID` rỗng | 243.007 dòng (22,8%) | RFM tính theo khách, không có mã thì không quy được cho ai. `groupby` sẽ **âm thầm** bỏ các dòng này |
| 2 | Đơn huỷ (`Invoice` bắt đầu `C`) | 19.494 dòng | Là bút toán đảo, không phải giao dịch mua. Frequency đếm sai |
| 3 | `Quantity <= 0` | 22.950 dòng | Monetary có thể âm, khiến `log1p` trả `NaN` và dừng toàn bộ pipeline |
| 4 | `Price <= 0` | 6.207 dòng | Bút toán kế toán. Giá nhỏ nhất là −53.594 bảng, không phải giá sản phẩm |
| 5 | Mã phi sản phẩm | ~5.900 dòng | `POST` xuất hiện ở hầu hết hóa đơn, sẽ sinh ra hàng loạt luật kết hợp vô nghĩa |
| 6 | Dòng trùng lặp trong cùng một sheet | 11.481 dòng | Thổi phồng Monetary |
| 7 | Hai sheet chồng lấn 01–09/12/2010 | 22.523 dòng (1.088 hóa đơn) | Giao dịch 9 ngày đó bị tính **hai lần** vào Monetary. `drop_duplicates()` **không bắt được** vì cột `SourceSheet` khác nhau |
| 8 | Đơn gốc của các đơn đã huỷ | 6.430 dòng, 712.074 bảng | Doanh thu ảo, tạo **outlier giả** (khách 16446, 12346) kéo lệch centroid |

Đáng lưu ý: chỉ vấn đề **số 3** gây lỗi dừng chương trình. Bảy vấn đề còn lại khiến
chương trình vẫn chạy bình thường nhưng cho kết quả sai — dạng lỗi khó phát hiện hơn.
Vấn đề 7 và 8 đặc biệt nguy hiểm vì chúng **sống sót qua các bước lọc thông thường**.

---

## 3. Làm sạch dữ liệu

### 3.1 Định nghĩa bộ lọc sản phẩm

In [14]:
def is_product(stockcode: pd.Series) -> pd.Series:
    """Mã sản phẩm hợp lệ: 5 chữ số (kèm chữ cái tùy chọn), hoặc hệ mã DCGS.

    Nhóm DCGS được giữ lại vì đã kiểm chứng ở mục 2.7 rằng đó là sản phẩm thật.
    """
    return (
        stockcode.str.match(r'^\d{5}[A-Za-z]*$')
        | stockcode.str.match(r'^DCGS')
    )

### 3.2 Pipeline lọc kèm nhật ký

Mỗi bước lọc được ghi lại số dòng còn lại, tạo thành bảng kiểm toán.
Bảng này cho phép người đọc kiểm chứng chính xác dữ liệu bị loại ở đâu và bao nhiêu.

**Thứ tự các bước có ý nghĩa:** phải bỏ đơn gốc bị huỷ **trước** khi bỏ dòng huỷ, vì
cần dòng huỷ để biết đơn nào bị huỷ. Phần chồng lấn giữa hai sheet được bỏ đầu tiên,
nếu không một dòng huỷ có thể bị ghép hai lần.

In [15]:
log = []

def step(name: str, data: pd.DataFrame) -> pd.DataFrame:
    log.append({'Bước': name, 'Còn lại': len(data)})
    return data


clean = step('Dữ liệu thô',                    df.copy())
clean = step('Bỏ phần chồng lấn giữa 2 sheet', clean[~dup_sheet])
clean = step('Bỏ Customer ID rỗng',            clean[clean['Customer ID'].notna()])
clean = step('Bỏ đơn gốc đã bị huỷ',           clean.drop(index=match_cancellations(clean)['buy_idx']))
clean = step('Bỏ dòng huỷ (Invoice có C)',     clean[~clean['Invoice'].str.startswith('C')])
clean = step('Bỏ Quantity <= 0',               clean[clean['Quantity'] > 0])
clean = step('Bỏ Price <= 0',                  clean[clean['Price'] > 0])
clean = step('Bỏ mã phi sản phẩm',             clean[is_product(clean['StockCode'])])
clean = step('Bỏ dòng trùng lặp',              clean.drop_duplicates())

# Customer ID là định danh, không phải đại lượng đo -> ép về số nguyên
clean['Customer ID'] = clean['Customer ID'].astype('int64')

# Doanh thu từng dòng, dùng để tính Monetary ở notebook sau
clean['TotalPrice'] = clean['Quantity'] * clean['Price']

audit = pd.DataFrame(log)
audit['Mất đi']    = -audit['Còn lại'].diff().fillna(0).astype(int)
audit['% còn lại'] = (audit['Còn lại'] / len(df) * 100).round(2)

print(audit.to_string(index=False))

                          Bước  Còn lại  Mất đi  % còn lại
                   Dữ liệu thô  1067371       0     100.00
Bỏ phần chồng lấn giữa 2 sheet  1044848   22523      97.89
           Bỏ Customer ID rỗng   809561  235287      75.85
          Bỏ đơn gốc đã bị huỷ   803131    6430      75.24
    Bỏ dòng huỷ (Invoice có C)   784685   18446      73.52
              Bỏ Quantity <= 0   784685       0      73.52
                 Bỏ Price <= 0   784616      69      73.51
            Bỏ mã phi sản phẩm   781932    2684      73.26
             Bỏ dòng trùng lặp   770451   11481      72.18


### 3.3 Kết quả sau làm sạch

In [16]:
print(f'Số dòng             : {len(clean):>8,}  ({len(clean)/len(df):.1%} dữ liệu gốc)')
print(f'Khách hàng duy nhất : {clean["Customer ID"].nunique():>8,}')
print(f'Hóa đơn duy nhất    : {clean["Invoice"].nunique():>8,}')
print(f'Mã sản phẩm duy nhất: {clean["StockCode"].nunique():>8,}')
print()
print('Kiểm tra: không còn giá trị bất thường')
print(f'  Quantity <= 0 : {(clean["Quantity"] <= 0).sum()}')
print(f'  Price <= 0    : {(clean["Price"] <= 0).sum()}')
print(f'  Customer ID NA: {clean["Customer ID"].isna().sum()}')
print()
print('Kiểm tra: hai outlier giả đã được sửa')
print(clean[clean['Customer ID'].isin([16446, 12346])]
      .groupby('Customer ID')['TotalPrice'].sum().round(2).to_string())
print()
clean.head()

Số dòng             :  770,451  (72.2% dữ liệu gốc)
Khách hàng duy nhất :    5,839
Hóa đơn duy nhất    :   36,324
Mã sản phẩm duy nhất:    4,611

Kiểm tra: không còn giá trị bất thường
  Quantity <= 0 : 0
  Price <= 0    : 0
  Customer ID NA: 0

Kiểm tra: hai outlier giả đã được sửa
Customer ID
12346    169.36
16446      2.90



,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010,30.0


In [17]:
clean.to_parquet(CLEAN_OUT, index=False)

size_raw   = XLSX.stat().st_size / 1024**2
size_clean = CLEAN_OUT.stat().st_size / 1024**2
print(f'Đã lưu {CLEAN_OUT}')
print(f'  Excel gốc : {size_raw:.1f} MB')
print(f'  Parquet   : {size_clean:.1f} MB')

Đã lưu data/retail_clean.parquet
  Excel gốc : 43.5 MB
  Parquet   : 5.9 MB


---

## 4. Tổng kết

**Đã hoàn thành**

- Hợp nhất 1.067.371 dòng từ hai sheet, xử lý ba cột bị kiểu hỗn hợp
- Khảo sát và định lượng tám vấn đề chất lượng dữ liệu
- Phát hiện hai lỗi **sống sót qua các bước lọc thông thường**:
  - hai sheet chồng lấn 9 ngày, làm 22.523 dòng bị tính hai lần
  - 6.430 dòng mua đã bị huỷ nhưng vẫn còn trong dữ liệu, tạo 712.074 bảng doanh thu ảo và
    các outlier giả như khách 16446 (168.000 bảng ảo)
- Kiểm chứng nhóm mã `DCGS` trước khi lọc, tránh loại nhầm sản phẩm hợp lệ
- Làm sạch theo pipeline có nhật ký: 770.451 dòng, 5.839 khách hàng, xuất `retail_clean.parquet`

**Hạn chế còn lại:** các dòng huỷ **một phần** đơn, hoặc huỷ đơn mua trước 12/2009, không
ghép được với đơn gốc nên chưa được trừ. Phần này nhỏ và không tạo outlier giả.

**Thông số chuyển sang notebook tiếp theo**

| Thông số | Vai trò |
|---|---|
| Ngày cuối cùng trong dữ liệu | Ngày tham chiếu Recency = ngày này + 1 |
| Số khách hàng sau làm sạch | Quyết định tính khả thi của Agglomerative Clustering, vốn cần bộ nhớ $O(n^2)$ |
| Cột `TotalPrice` | Dùng để tính Monetary |

**Bước tiếp theo:** `02_rfm.ipynb` — tính RFM, xử lý độ lệch bằng `log1p`,
chuẩn hóa bằng `StandardScaler`.